In [1]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# 1. قراءة البيانات
train_df = pd.read_parquet("../artifacts/train.parquet")
val_df = pd.read_parquet("../artifacts/val.parquet")
test_df = pd.read_parquet("../artifacts/test.parquet")

# 2. دالة استخراج الفيتشرز الزمنية والدلتاس
def extract_features(df):
    df = df.copy()
    date_cols = [
        "order_purchase_timestamp", "order_approved_at",
        "order_delivered_carrier_date", "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])

    # Time features (معلومات الموعد وقت الشراء فقط)
    df["purchase_year"] = df["order_purchase_timestamp"].dt.year
    df["purchase_month"] = df["order_purchase_timestamp"].dt.month
    df["purchase_dayofweek"] = df["order_purchase_timestamp"].dt.dayofweek
    df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour

    # Time deltas
    df["estimated_delivery_days"] = (
        df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
    ).dt.total_seconds() / (24 * 3600)

    return df

train_feats = extract_features(train_df)
val_feats = extract_features(val_df)
test_feats = extract_features(test_df)

# 3. تحديد أعمدة الفيتشرز المطلوبة للتدريب
feature_cols = [
    "total_items", "total_price", "total_freight", "sellers_count",
    "total_payment_value", "payment_types_count", "max_installments",
    "purchase_year", "purchase_month", "purchase_dayofweek", "purchase_hour",
    "estimated_delivery_days"
]

X_train = train_feats[feature_cols]
y_train = train_feats["is_late"]

X_val = val_feats[feature_cols]
y_val = val_feats["is_late"]

X_test = test_feats[feature_cols]
y_test = test_feats["is_late"]

# 4. Imputation + Scaling (Fit على Train وتطبيق على الكل)
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

# Fit on Train only!
X_train_imputed = imputer.fit_transform(X_train)
X_train_scaled = scaler.fit_transform(X_train_imputed)

# Transform Val & Test
X_val_scaled = scaler.transform(imputer.transform(X_val))
X_test_scaled = scaler.transform(imputer.transform(X_test))

# 5. حفظ الـ Artifacts (البيانات الجاهزة والمجسمات المدربة)
os.makedirs("../artifacts/models", exist_ok=True)

# حفظ الـ Transformers
joblib.dump(imputer, "../artifacts/models/imputer.joblib")
joblib.dump(scaler, "../artifacts/models/scaler.joblib")
joblib.dump(feature_cols, "../artifacts/models/feature_cols.joblib")

# حفظ الجداول المعالجة
pd.DataFrame(X_train_scaled, columns=feature_cols).assign(is_late=y_train.values).to_parquet("../artifacts/train_processed.parquet")
pd.DataFrame(X_val_scaled, columns=feature_cols).assign(is_late=y_val.values).to_parquet("../artifacts/val_processed.parquet")
pd.DataFrame(X_test_scaled, columns=feature_cols).assign(is_late=y_test.values).to_parquet("../artifacts/test_processed.parquet")

print("تم تجهيز وحفظ الفيتشرز والـ Transformers بنجاح!")

تم تجهيز وحفظ الفيتشرز والـ Transformers بنجاح!
